# Gaeta–Lie Neural SDE — 50D SDEs (Ex 50.1–50.3)

**Paper:** *Gaeta–Lie Neural SDEs: Symmetry-Regularized Learning of Stochastic Dynamics* (NeurIPS 2026) — Sections 5.1–5.2, Figure 2, Table 1, Appendices G–I (Tables 12–14).

Full pipeline with the R = 333 library (low-rank rotations/shears) and m = 50 generators. Paper examples: `"ring_double_well_50d"` (Ex 50.1), `"low_rank_ou_50d"` (Ex 50.2), `"tanh_chain_50d"` (Ex 50.3). Use `N_TRAJECTORIES = 100` for the data-efficient regime and `60` for the data-scarce regime.

**Pipeline naming used in the code:** Stage I = baseline neural SDE surrogate; Stage II = symmetry discovery over the basis library; Stage F = Stage III (symmetry-regularized refinement).

Run the cells top to bottom. All configuration lives in `CONFIG` in the first code cell. See the [repository README](../../README.md) for installation and the full experiment index.

## Cell 1 — Config, imports, global knobs

In [ ]:
# ============================================================
# Cell 1 — Config, imports, global knobs
# ============================================================

from jax import config as jax_config
jax_config.update("jax_enable_x64", False)

import math
import time
from functools import partial

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

print("JAX version:", jax.__version__)
print("JAX devices:", jax.devices())


# ----------------------------
# Main configuration knobs
# ----------------------------
CONFIG = {
    # Reproducibility
    "SEED": 42,

    # ----------------------------
    # 50D synthetic SDE generation
    # ----------------------------
    "STATE_DIM": 50,

    # Choose one:
    #   "ring_double_well_50d"
    #   "low_rank_ou_50d"
    #   "block_lorenz_50d"
    #   "tanh_chain_50d"
    "TRUE_SDE_NAME": "ring_double_well_50d",

    "T_FINAL": 2.0,
    "DT": 0.01,
    "N_TRAJECTORIES": 64,
    "X0_MEAN": 0.0,
    "X0_STD": 1.0,
    "TRAIN_FRACTION": 0.8,

    # ----------------------------
    # Neural SDE surrogate
    # Input:  (t, x1, ..., x50), dimension 51
    # Output: (f1,...,f50, raw_sigma1,...,raw_sigma50), dimension 100
    # ----------------------------
    "MLP_HIDDEN_DIM": 192,
    "MLP_DEPTH": 3,
    "INPUT_T_SCALE": 2.0,
    "INPUT_X_SCALE": 3.0,
    "SIGMA_EPS": 1e-4,

    # ----------------------------
    # Baseline surrogate training
    # ----------------------------
    "BASELINE_EPOCHS": 2000,
    "BASELINE_BATCH_SIZE": 512,
    "BASELINE_LR": 2e-3,
    "PRINT_EVERY": 250,

    # ----------------------------
    # Symmetry discovery
    #
    # For 50D full-rank diffusion, the theoretical n+2 upper bound is 52.
    # Do not sweep all 52 initially. Start with 1, then try 2, 3, 5.
    # ----------------------------
    "N_GENERATORS": 50,
    "SYM_DISCOVERY_EPOCHS": 4000,
    "SYM_DISCOVERY_LR": 5e-3,
    "SYM_DISCOVERY_POINTS": 256,
    "SYM_DISCOVERY_T_BOX": (0.0, 2.0),
    "SYM_DISCOVERY_X_BOX": (-3.0, 3.0),
    "SYM_INIT_SCALE": 0.03,

    # Symmetry-discovery loss weights
    "LAMBDA_DET": 1.0,
    "LAMBDA_CLOSE": 0.5,
    "LAMBDA_IND": 0.05,
    "LAMBDA_PROJECTABLE": 1.0,
    "LAMBDA_NORM_HINGE": 0.01,
    "MIN_GENERATOR_NORM": 0.10,
    "CLOSURE_RCOND": 1e-5,

    # 50D basis mode
    # Options:
    #   "minimal_104"
    #   "expanded_gaeta_lowrank"
    "BASIS_MODE": "expanded_gaeta_lowrank",

    # Gaeta-style exponential linear-drift basis rates.
    # Adds:
    #   e^{mu t} ∂x_i
    #   e^{2mu t}(∂t + mu x·∇x)
    #
    # In code we use normalized time t/T for numerical stability.
    "BASIS_LINEAR_MU_LIST": [-1.0, -0.5, 0.5, 1.0],
    "BASIS_EXP_CLIP": 6.0,

    # Low-rank linear basis directions.
    # K=6 gives:
    #   rotations: K(K-1)/2 = 15
    #   bidirectional adjacent shears: 2(K-1) = 10
    "BASIS_LOW_RANK_K": 6,

    "BASIS_ADD_EXPONENTIAL": True,
    "BASIS_ADD_LOW_RANK_ROTATIONS": True,
    "BASIS_ADD_LOW_RANK_SHEARS": True,

    # ----------------------------
    # Stage F: symmetry-regularized surrogate refinement
    # ----------------------------
    "STAGE_F_EPOCHS": 800,
    "STAGE_F_BATCH_SIZE": 512,
    "STAGE_F_LR": 3e-4,
    "STAGE_F_LAMBDA_SYM_DET": 105000,
    "STAGE_F_LAMBDA_PROJECTABLE": 1e-3,

    # Extra collocation points for Stage F symmetry loss
    "STAGE_F_N_EXTRA_SYM_POINTS": 128,

    # Only use a small subset of trajectory batch points for symmetry loss.
    # This prevents the 50D Jacobian computations from becoming too heavy.
    "STAGE_F_N_DATA_SYM_POINTS": 128,

    "STAGE_F_EXTRA_T_BOX": (0.0, 2.0),
    "STAGE_F_EXTRA_X_BOX": (-3.0, 3.0),

    # Diagnostics
    "N_EVAL_POINTS": 2000,
    "N_PLOT_GRID": 160,

    # Plot only these components in 50D
    "PLOT_COMPONENTS": [0, 1, 2, 9, 24, 49],
}

D = CONFIG["STATE_DIM"]
Q = D + 1  # coordinates are z = (t, x1, ..., x50)

key = jax.random.PRNGKey(CONFIG["SEED"])

## Cell 2 — Editable 50D SDE registry and synthetic data

In [ ]:
# ============================================================
# Cell 2 — Editable 50D SDE registry and synthetic data
# ============================================================

# ------------------------------------------------------------------
# How to use:
#
# Change CONFIG["TRUE_SDE_NAME"] in Cell 1.
#
# Available examples:
#   "ring_double_well_50d"
#   "low_rank_ou_50d"
#   "block_lorenz_50d"
#   "tanh_chain_50d"
#
# Each SDE entry defines:
#   drift_fn(t, x) -> shape (..., 50)
#   sigma_fn(t, x) -> shape (..., 50), positive diagonal diffusion
# ------------------------------------------------------------------


def _idx_float():
    return jnp.arange(D, dtype=jnp.float32)


def _phase_vector():
    return 2.0 * jnp.pi * _idx_float() / float(D)


def _mode_sin(k):
    return jnp.sin(2.0 * jnp.pi * k * _idx_float() / float(D))


def _mode_cos(k):
    return jnp.cos(2.0 * jnp.pi * k * _idx_float() / float(D))


def _normalize_mode(v):
    return v / (jnp.linalg.norm(v) + 1e-8)


# ============================================================
# 50D SDE example 1: ring-coupled double-well
# ============================================================

def drift_ring_double_well_50d(t, x):
    """
    50D nonlinear ring-coupled double-well SDE.

    Drift:
        f_i = x_i - x_i^3
              + c (x_{i-1} + x_{i+1} - 2 x_i)
              + weak time forcing_i(t)

    Nontrivial features:
        - nonlinear bistability in every coordinate
        - local spatial coupling on a ring
        - weak heterogeneous forcing
    """
    c = 0.25
    lap = jnp.roll(x, 1, axis=-1) + jnp.roll(x, -1, axis=-1) - 2.0 * x

    phase = _phase_vector()
    forcing = 0.08 * jnp.sin(2.0 * jnp.pi * t + phase)

    return x - x**3 + c * lap + forcing


def sigma_ring_double_well_50d(t, x):
    """
    Positive state-dependent diagonal diffusion.
    """
    phase = _phase_vector()
    base = 0.18 + 0.02 * jnp.sin(phase) ** 2
    return base + 0.04 * jax.nn.sigmoid(x)


# ============================================================
# 50D SDE example 2: low-rank coupled OU/spiral system
# ============================================================

def drift_low_rank_ou_50d(t, x):
    """
    Stable 50D linear/nonlinear low-rank OU-like SDE.

    Drift:
        stable coordinate damping
        + skew ring coupling
        + low-rank global coupling
        + weak periodic forcing
        + small tanh nonlinearity

    This is useful because the true drift has global low-rank structure
    but is still learned by a generic neural SDE.
    """
    idx = _idx_float()

    # Coordinate-dependent damping
    lam = 0.45 + 0.30 * (idx / float(D - 1))

    # Ring skew/transport-like coupling
    skew = 0.18 * (jnp.roll(x, -1, axis=-1) - jnp.roll(x, 1, axis=-1))

    v1 = _normalize_mode(_mode_sin(1))
    v2 = _normalize_mode(_mode_cos(2))
    v3 = _normalize_mode(_mode_sin(3))

    p1 = jnp.sum(x * v1, axis=-1, keepdims=True)
    p2 = jnp.sum(x * v2, axis=-1, keepdims=True)
    p3 = jnp.sum(x * v3, axis=-1, keepdims=True)

    low_rank = (
        0.45 * p1 * v2
        - 0.35 * p2 * v1
        + 0.25 * jnp.tanh(p3) * v3
    )

    forcing = 0.05 * (
        jnp.sin(1.2 * t) * v1
        + jnp.cos(0.8 * t) * v2
    )

    return -lam * x + skew + low_rank + forcing + 0.05 * jnp.tanh(x)


def sigma_low_rank_ou_50d(t, x):
    v1 = _normalize_mode(_mode_sin(1))
    p1 = jnp.sum(x * v1, axis=-1, keepdims=True)
    base = 0.20 + 0.03 * jnp.sin(_phase_vector()) ** 2
    return base + 0.03 * jax.nn.sigmoid(p1) + 0.02 * jnp.tanh(x) ** 2


# ============================================================
# 50D SDE example 3: block Lorenz-style system
# ============================================================

def drift_block_lorenz_50d(t, x):
    """
    50D block-tamed Lorenz-style SDE.

    First 48 coordinates form 16 weakly coupled Lorenz-like triples.
    Last 2 coordinates are stable auxiliary variables coupled to block means.

    The Lorenz blocks are tamed to avoid numerical explosions.
    """
    x_blocks = x[..., :48].reshape(x.shape[:-1] + (16, 3))
    y = x[..., 48:50]

    x1 = x_blocks[..., 0]
    x2 = x_blocks[..., 1]
    x3 = x_blocks[..., 2]

    sigma_l = 4.0
    rho = 8.0
    beta = 8.0 / 3.0

    g1 = sigma_l * (x2 - x1)
    g2 = x1 * (rho - x3) - x2
    g3 = x1 * x2 - beta * x3

    raw = jnp.stack([g1, g2, g3], axis=-1)

    norm2 = jnp.sum(x_blocks**2, axis=-1, keepdims=True)
    tame = 1.0 + 0.05 * norm2

    block_f = raw / tame

    # Weak coupling between neighboring blocks
    block_lap = (
        jnp.roll(x_blocks, 1, axis=-2)
        + jnp.roll(x_blocks, -1, axis=-2)
        - 2.0 * x_blocks
    )
    block_f = block_f + 0.05 * block_lap

    # Auxiliary coordinates
    block_mean = jnp.mean(x_blocks, axis=(-2, -1), keepdims=False)
    f_y1 = -0.60 * y[..., 0] + 0.10 * jnp.tanh(block_mean)
    f_y2 = -0.75 * y[..., 1] + 0.10 * jnp.sin(t)

    f_y = jnp.stack([f_y1, f_y2], axis=-1)

    return jnp.concatenate([block_f.reshape(x.shape[:-1] + (48,)), f_y], axis=-1)


def sigma_block_lorenz_50d(t, x):
    base = 0.12 + 0.02 * jnp.sin(_phase_vector()) ** 2
    return base + 0.03 * jnp.tanh(x) ** 2


# ============================================================
# 50D SDE example 4: nonlinear tanh reaction chain
# ============================================================

def drift_tanh_chain_50d(t, x):
    """
    50D smooth nonlinear reaction-chain SDE.

    Drift:
        damping
        + forward/backward tanh couplings
        + weak global low-rank feedback
        + heterogeneous forcing

    This is bounded and stable, but still nonlinear and coupled.
    """
    phase = _phase_vector()

    f_local = (
        -0.55 * x
        + 0.35 * jnp.tanh(jnp.roll(x, -1, axis=-1))
        - 0.25 * jnp.tanh(jnp.roll(x, 1, axis=-1))
        + 0.10 * jnp.tanh(jnp.roll(x, -2, axis=-1))
    )

    v1 = _normalize_mode(_mode_cos(1))
    v2 = _normalize_mode(_mode_sin(2))

    p1 = jnp.sum(x * v1, axis=-1, keepdims=True)
    p2 = jnp.sum(x * v2, axis=-1, keepdims=True)

    global_feedback = 0.20 * jnp.tanh(p1) * v2 - 0.15 * jnp.tanh(p2) * v1

    forcing = 0.04 * jnp.sin(1.5 * t + phase)

    return f_local + global_feedback + forcing


def sigma_tanh_chain_50d(t, x):
    return 0.22 + 0.04 * jnp.sin(x) ** 2


# ============================================================
# Registry
# ============================================================

SDE_EXAMPLES = {
    "ring_double_well_50d": {
        "description": "50D nonlinear ring-coupled double-well SDE with state-dependent diffusion.",
        "drift_fn": drift_ring_double_well_50d,
        "sigma_fn": sigma_ring_double_well_50d,
        "t_final": 2.0,
        "dt": 0.01,
        "x0_mean": 0.0,
        "x0_std": 0.75,
        "input_x_scale": 2.5,
        "x_box": (-2.5, 2.5),
    },

    "low_rank_ou_50d": {
        "description": "50D stable low-rank OU/spiral SDE with global modes and state-dependent diffusion.",
        "drift_fn": drift_low_rank_ou_50d,
        "sigma_fn": sigma_low_rank_ou_50d,
        "t_final": 2.0,
        "dt": 0.01,
        "x0_mean": 0.0,
        "x0_std": 1.0,
        "input_x_scale": 3.0,
        "x_box": (-3.0, 3.0),
    },

    "block_lorenz_50d": {
        "description": "50D tamed Lorenz-block SDE: 16 coupled triples plus 2 stable auxiliary coordinates.",
        "drift_fn": drift_block_lorenz_50d,
        "sigma_fn": sigma_block_lorenz_50d,
        "t_final": 1.0,
        "dt": 0.005,
        "x0_mean": 0.0,
        "x0_std": 0.8,
        "input_x_scale": 5.0,
        "x_box": (-5.0, 8.0),
    },

    "tanh_chain_50d": {
        "description": "50D damped nonlinear tanh reaction chain with low-rank feedback.",
        "drift_fn": drift_tanh_chain_50d,
        "sigma_fn": sigma_tanh_chain_50d,
        "t_final": 3.0,
        "dt": 0.01,
        "x0_mean": 0.0,
        "x0_std": 1.0,
        "input_x_scale": 3.0,
        "x_box": (-3.0, 3.0),
    },
}


def activate_sde_from_config():
    """
    Select active SDE from CONFIG["TRUE_SDE_NAME"] and update
    dependent CONFIG fields automatically.
    """
    name = CONFIG["TRUE_SDE_NAME"]

    if name not in SDE_EXAMPLES:
        raise ValueError(
            f"Unknown TRUE_SDE_NAME={name}. "
            f"Available choices: {list(SDE_EXAMPLES.keys())}"
        )

    spec = SDE_EXAMPLES[name]

    CONFIG["T_FINAL"] = float(spec["t_final"])
    CONFIG["DT"] = float(spec["dt"])
    CONFIG["X0_MEAN"] = spec["x0_mean"]
    CONFIG["X0_STD"] = spec["x0_std"]
    CONFIG["INPUT_T_SCALE"] = float(spec["t_final"])
    CONFIG["INPUT_X_SCALE"] = float(spec["input_x_scale"])

    x_box = tuple(float(v) for v in spec["x_box"])

    CONFIG["SYM_DISCOVERY_T_BOX"] = (0.0, float(spec["t_final"]))
    CONFIG["STAGE_F_EXTRA_T_BOX"] = (0.0, float(spec["t_final"]))

    CONFIG["SYM_DISCOVERY_X_BOX"] = x_box
    CONFIG["STAGE_F_EXTRA_X_BOX"] = x_box

    return spec


ACTIVE_SDE = activate_sde_from_config()


def _as_50d_vector(value, name):
    """
    Convert scalar or length-D array/tuple into shape (D,).
    """
    arr = jnp.asarray(value, dtype=jnp.float32)

    if arr.ndim == 0:
        return jnp.full((D,), arr, dtype=jnp.float32)

    if arr.shape != (D,):
        raise ValueError(f"{name} must be scalar or shape ({D},), got {arr.shape}")

    return arr


def true_drift(t, x):
    """
    Dispatch to active 50D drift.

    x shape:
        (..., 50)

    return shape:
        (..., 50)
    """
    return ACTIVE_SDE["drift_fn"](t, x)


def true_sigma(t, x):
    """
    Dispatch to active 50D diagonal diffusion.

    x shape:
        (..., 50)

    return shape:
        (..., 50)
    """
    sigma = ACTIVE_SDE["sigma_fn"](t, x)
    return jnp.maximum(sigma, 1e-6)


def print_active_sde_report():
    print("\n" + "=" * 72)
    print("ACTIVE 50D SYNTHETIC SDE")
    print("=" * 72)
    print(f"TRUE_SDE_NAME: {CONFIG['TRUE_SDE_NAME']}")
    print(f"Description  : {ACTIVE_SDE['description']}")
    print(f"STATE_DIM    : {D}")
    print(f"T_FINAL      : {CONFIG['T_FINAL']}")
    print(f"DT           : {CONFIG['DT']}")
    print(f"X0_MEAN      : {CONFIG['X0_MEAN']}")
    print(f"X0_STD       : {CONFIG['X0_STD']}")
    print(f"X_BOX        : {CONFIG['STAGE_F_EXTRA_X_BOX']}")
    print(f"INPUT_T_SCALE: {CONFIG['INPUT_T_SCALE']}")
    print(f"INPUT_X_SCALE: {CONFIG['INPUT_X_SCALE']}")
    print("=" * 72)


print_active_sde_report()


@jax.jit
def simulate_sde_paths(key):
    """
    Simulate all trajectories using Euler-Maruyama.

    Returns
    -------
    times : shape (n_steps + 1,)
    paths : shape (n_paths, n_steps + 1, 50)
    """
    T = CONFIG["T_FINAL"]
    dt = CONFIG["DT"]
    n_steps = int(round(T / dt))
    n_paths = CONFIG["N_TRAJECTORIES"]

    key_x0, key_noise = jax.random.split(key)

    x0_mean = _as_50d_vector(CONFIG["X0_MEAN"], "X0_MEAN")
    x0_std = _as_50d_vector(CONFIG["X0_STD"], "X0_STD")

    x0 = x0_mean[None, :] + x0_std[None, :] * jax.random.normal(
        key_x0,
        (n_paths, D),
    )

    noise = jax.random.normal(key_noise, (n_steps, n_paths, D))

    times = jnp.arange(n_steps + 1, dtype=jnp.float32) * dt
    sqrt_dt = jnp.sqrt(dt)

    def step(x, inp):
        t, zeta = inp
        f = true_drift(t, x)
        s = true_sigma(t, x)
        x_next = x + f * dt + s * sqrt_dt * zeta
        return x_next, x_next

    _, xs_tail = jax.lax.scan(step, x0, (times[:-1], noise))

    paths = jnp.concatenate([x0[None, :, :], xs_tail], axis=0)
    paths = jnp.transpose(paths, (1, 0, 2))

    return times, paths


def make_increment_dataset(times, paths):
    """
    Convert simulated trajectories into one-step increment data.

    Returns:
        t0 : shape (n_samples,)
        x0 : shape (n_samples, 50)
        dt : shape (n_samples,)
        dx : shape (n_samples, 50)
    """
    n_paths, n_times, dim = paths.shape
    n_steps = n_times - 1
    dt = CONFIG["DT"]

    t0 = jnp.broadcast_to(times[:-1][None, :], (n_paths, n_steps)).reshape(-1)
    x0 = paths[:, :-1, :].reshape(-1, dim)
    dx = (paths[:, 1:, :] - paths[:, :-1, :]).reshape(-1, dim)
    dt_arr = jnp.full((x0.shape[0],), dt, dtype=jnp.float32)

    return t0, x0, dt_arr, dx


def train_test_split_dataset(key, dataset, train_fraction):
    n = dataset[0].shape[0]
    perm = jax.random.permutation(key, n)
    n_train = int(train_fraction * n)

    idx_train = perm[:n_train]
    idx_test = perm[n_train:]

    train = tuple(arr[idx_train] for arr in dataset)
    test = tuple(arr[idx_test] for arr in dataset)

    return train, test


def sample_batch(key, dataset, batch_size):
    n = dataset[0].shape[0]
    idx = jax.random.randint(key, (batch_size,), 0, n)
    return tuple(arr[idx] for arr in dataset)


# ----------------------------
# Generate data
# ----------------------------
key, subkey = jax.random.split(key)
times, paths = simulate_sde_paths(subkey)
dataset = make_increment_dataset(times, paths)

key, subkey = jax.random.split(key)
train_data, test_data = train_test_split_dataset(
    subkey,
    dataset,
    CONFIG["TRAIN_FRACTION"],
)

print("times shape:", times.shape)
print("paths shape:", paths.shape)
print("train samples:", train_data[0].shape[0])
print("test samples:", test_data[0].shape[0])


# ----------------------------
# Quick trajectory plots for selected components only
# ----------------------------
for comp in CONFIG["PLOT_COMPONENTS"]:
    plt.figure(figsize=(8, 4))
    for i in range(min(12, CONFIG["N_TRAJECTORIES"])):
        plt.plot(np.array(times), np.array(paths[i, :, comp]), alpha=0.6)
    plt.xlabel("t")
    plt.ylabel(f"X_{comp+1}(t)")
    plt.title(
        f"{CONFIG['TRUE_SDE_NAME']} trajectories: component {comp+1}"
    )
    plt.grid(True)
    plt.show()


# ----------------------------
# Sanity check drift/diffusion values
# ----------------------------
key, subkey = jax.random.split(key)
x_probe = jax.random.normal(subkey, (8, D))
t_probe = 0.5 * CONFIG["T_FINAL"]

print("\nDrift probe shape:", true_drift(t_probe, x_probe).shape)
print("Sigma probe shape:", true_sigma(t_probe, x_probe).shape)
print("Sigma probe min  :", float(jnp.min(true_sigma(t_probe, x_probe))))

## Cell 3 — 50D Neural SDE surrogate, Adam optimizer, baseline training

In [ ]:
# ============================================================
# Cell 3 — 50D Neural SDE surrogate, Adam optimizer, baseline training
# ============================================================

def glorot_uniform(key, in_dim, out_dim):
    lim = jnp.sqrt(6.0 / (in_dim + out_dim))
    return jax.random.uniform(key, (in_dim, out_dim), minval=-lim, maxval=lim)


class MLP:
    """
    Small pure-JAX MLP class.

    For the 3D SDE:
        input  = (t, x1, x2, x3), dimension 4
        output = (f1, f2, f3, raw_sigma1, raw_sigma2, raw_sigma3), dimension 6
    """

    def __init__(self, in_dim=4, hidden_dim=96, depth=3, out_dim=6):
        self.in_dim = in_dim
        self.hidden_dim = hidden_dim
        self.depth = depth
        self.out_dim = out_dim

    def init(self, key):
        sizes = [self.in_dim] + [self.hidden_dim] * self.depth + [self.out_dim]
        keys = jax.random.split(key, len(sizes) - 1)

        params = []
        for k, in_d, out_d in zip(keys, sizes[:-1], sizes[1:]):
            W = glorot_uniform(k, in_d, out_d)
            b = jnp.zeros((out_d,), dtype=jnp.float32)
            params.append({"W": W, "b": b})
        return params

    def apply(self, params, z):
        h = z
        for layer in params[:-1]:
            h = jnp.tanh(h @ layer["W"] + layer["b"])
        out = h @ params[-1]["W"] + params[-1]["b"]
        return out


surrogate_model = MLP(
    in_dim=Q,
    hidden_dim=CONFIG["MLP_HIDDEN_DIM"],
    depth=CONFIG["MLP_DEPTH"],
    out_dim=2 * D,
)


def normalize_input(t, x):
    tn = jnp.asarray([t / CONFIG["INPUT_T_SCALE"]], dtype=jnp.float32)
    xn = x / CONFIG["INPUT_X_SCALE"]
    return jnp.concatenate([tn, xn], axis=0)


def surrogate_f_sigma(params, t, x):
    z = normalize_input(t, x)
    raw = surrogate_model.apply(params, z)

    f = raw[:D]
    sigma = jax.nn.softplus(raw[D:]) + CONFIG["SIGMA_EPS"]
    return f, sigma


def f_vec_single(params, z):
    """
    z = (t, x1, x2, x3)
    returns f(t,x), shape (3,)
    """
    return surrogate_f_sigma(params, z[0], z[1:])[0]


def sigma_vec_single(params, z):
    """
    z = (t, x1, x2, x3)
    returns diagonal sigma(t,x), shape (3,)
    """
    return surrogate_f_sigma(params, z[0], z[1:])[1]


def data_nll_loss(params, batch):
    """
    Euler-Maruyama diagonal-Gaussian negative log-likelihood.

    batch = (t0, x0, dt, dx)
        t0 shape (B,)
        x0 shape (B, 3)
        dt shape (B,)
        dx shape (B, 3)
    """
    t0, x0, dt, dx = batch

    f, sigma = jax.vmap(lambda tt, xx: surrogate_f_sigma(params, tt, xx))(t0, x0)

    mean = f * dt[:, None]
    var = sigma**2 * dt[:, None] + 1e-8

    nll_components = 0.5 * (
        (dx - mean) ** 2 / var
        + jnp.log(2.0 * jnp.pi * var)
    )

    return jnp.mean(jnp.sum(nll_components, axis=-1))


# ----------------------------
# Lightweight pure-JAX Adam
# ----------------------------
def adam_init(params):
    m = jax.tree_util.tree_map(jnp.zeros_like, params)
    v = jax.tree_util.tree_map(jnp.zeros_like, params)
    t = jnp.array(0, dtype=jnp.int32)
    return {"m": m, "v": v, "t": t}


def adam_update(params, grads, state, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    t_new = state["t"] + 1

    m_new = jax.tree_util.tree_map(
        lambda m, g: beta1 * m + (1.0 - beta1) * g,
        state["m"],
        grads,
    )
    v_new = jax.tree_util.tree_map(
        lambda v, g: beta2 * v + (1.0 - beta2) * (g * g),
        state["v"],
        grads,
    )

    m_hat = jax.tree_util.tree_map(lambda m: m / (1.0 - beta1**t_new), m_new)
    v_hat = jax.tree_util.tree_map(lambda v: v / (1.0 - beta2**t_new), v_new)

    params_new = jax.tree_util.tree_map(
        lambda p, m, v: p - lr * m / (jnp.sqrt(v) + eps),
        params,
        m_hat,
        v_hat,
    )

    return params_new, {"m": m_new, "v": v_new, "t": t_new}


@jax.jit
def baseline_train_step(params, opt_state, batch):
    loss, grads = jax.value_and_grad(data_nll_loss)(params, batch)
    params, opt_state = adam_update(
        params,
        grads,
        opt_state,
        CONFIG["BASELINE_LR"],
    )
    return params, opt_state, loss


# ----------------------------
# Train baseline surrogate
# ----------------------------
key, subkey = jax.random.split(key)
sde_params = surrogate_model.init(subkey)
sde_opt_state = adam_init(sde_params)

baseline_losses = []
t_start = time.time()

for epoch in range(1, CONFIG["BASELINE_EPOCHS"] + 1):
    key, subkey = jax.random.split(key)
    batch = sample_batch(subkey, train_data, CONFIG["BASELINE_BATCH_SIZE"])

    sde_params, sde_opt_state, loss = baseline_train_step(
        sde_params,
        sde_opt_state,
        batch,
    )

    baseline_losses.append(float(loss))

    if epoch % CONFIG["PRINT_EVERY"] == 0 or epoch == 1:
        train_loss = float(data_nll_loss(sde_params, train_data))
        test_loss = float(data_nll_loss(sde_params, test_data))
        print(
            f"[Baseline] epoch {epoch:5d} | "
            f"batch nll {float(loss): .5f} | "
            f"train nll {train_loss: .5f} | "
            f"test nll {test_loss: .5f}"
        )

print(f"Baseline training time: {time.time() - t_start:.2f} sec")

## Cell 4 — Editable 50D Gaeta-style expanded symmetry-basis library

In [ ]:
# ============================================================
# Cell 4 — Editable 50D Gaeta-style expanded symmetry-basis library
# ============================================================

# ------------------------------------------------------------------
# Each basis returns a 51-vector:
#     [tau, xi1, ..., xi50]
#
# Minimal 104-basis:
#   3 time-only generators:
#       ∂t, t∂t, t^2∂t
#   50 coordinate translations:
#       ∂x_i
#   50 coordinate scalings:
#       x_i ∂x_i
#   1 global Brownian/Gaeta scaling:
#       2(t/T)∂t + x·∇x
#
# Expanded Gaeta/low-rank additions:
#   1. Exponential translations:
#       e^{mu t/T} ∂x_i
#
#   2. Exponential projective linear-drift generators:
#       e^{2mu t/T}(∂t + mu x·∇x)
#
#   3. Low-rank rotations:
#       (v_b·x)v_a·∇x - (v_a·x)v_b·∇x
#
#   4. Low-rank shears:
#       (v_b·x)v_a·∇x
#
# The added basis elements are all at most linear in x, so xi_xx = 0.
# Therefore the Hessian-free determining-equation implementation remains valid.
# ------------------------------------------------------------------

BASIS_LIBRARY = []


def add_basis(name, source, fn):
    BASIS_LIBRARY.append({"name": name, "source": source, "fn": fn})


def time_basis(t):
    """
    Normalized time variable.

    We use t/T rather than raw t inside exponentials and polynomial
    time basis functions for numerical stability.
    """
    return t / CONFIG["INPUT_T_SCALE"]


def safe_exp(z):
    c = CONFIG["BASIS_EXP_CLIP"]
    return jnp.exp(jnp.clip(z, -c, c))


def zero_xi():
    return jnp.zeros((D,), dtype=jnp.float32)


def e_i(i):
    v = jnp.zeros((D,), dtype=jnp.float32)
    return v.at[i].set(1.0)


def unit(v):
    return v / (jnp.linalg.norm(v) + 1e-8)


def make_low_rank_directions(K):
    """
    Generic low-rank directions in coordinate-index space.

    These are not fitted from the data and do not use the true SDE.
    They are fixed Fourier-like directions:
        constant, sin(1), cos(1), sin(2), cos(2), ...
    """
    idx = jnp.arange(D, dtype=jnp.float32)
    dirs = []

    # Global mean direction.
    dirs.append(unit(jnp.ones((D,), dtype=jnp.float32)))

    freq = 1
    while len(dirs) < K:
        phase = 2.0 * jnp.pi * freq * (idx + 0.5) / float(D)

        dirs.append(unit(jnp.sin(phase)))

        if len(dirs) < K:
            dirs.append(unit(jnp.cos(phase)))

        freq += 1

    return tuple(dirs[:K])


# ============================================================
# Group A: time-only projectable basis
# ============================================================

add_basis(
    "time_translation__dt",
    "Generic projectable time translation",
    lambda t, x: jnp.concatenate(
        [jnp.array([1.0], dtype=jnp.float32), zero_xi()]
    ),
)

add_basis(
    "time_scaling__t_dt",
    "Generic projectable time scaling",
    lambda t, x: jnp.concatenate(
        [jnp.array([time_basis(t)], dtype=jnp.float32), zero_xi()]
    ),
)

add_basis(
    "time_quadratic__t2_dt",
    "Generic projectable time quadratic",
    lambda t, x: jnp.concatenate(
        [jnp.array([time_basis(t) ** 2], dtype=jnp.float32), zero_xi()]
    ),
)


# ============================================================
# Group B: coordinate translations ∂x_i
# ============================================================

for i in range(D):
    add_basis(
        f"translation__dx{i+1}",
        "Generic coordinate translation",
        lambda t, x, i=i: jnp.concatenate(
            [jnp.array([0.0], dtype=jnp.float32), e_i(i)]
        ),
    )


# ============================================================
# Group C: coordinate scalings x_i ∂x_i
# ============================================================

for i in range(D):
    add_basis(
        f"scaling__x{i+1}_dx{i+1}",
        "Generic coordinate scaling",
        lambda t, x, i=i: jnp.concatenate(
            [
                jnp.array([0.0], dtype=jnp.float32),
                jnp.zeros((D,), dtype=jnp.float32).at[i].set(x[i]),
            ]
        ),
    )


# ============================================================
# Group D: global Brownian/Gaeta lifted scaling
# ============================================================

add_basis(
    "global_brownian_scaling__2t_dt_plus_x_dot_dx",
    "50D lift of Brownian/Gaeta scaling",
    lambda t, x: jnp.concatenate(
        [jnp.array([2.0 * time_basis(t)], dtype=jnp.float32), x]
    ),
)


# ============================================================
# Group E: Gaeta-style exponential linear-drift basis
# ============================================================

if CONFIG.get("BASIS_ADD_EXPONENTIAL", False):
    for mu in CONFIG["BASIS_LINEAR_MU_LIST"]:
        mu = float(mu)

        if abs(mu) < 1e-12:
            continue

        # e^{mu t/T} ∂x_i
        for i in range(D):
            add_basis(
                f"exp_translation__mu_{mu:+.2f}__e_mut_dx{i+1}",
                "Gaeta-style exponential translation e^{mu t} ∂x_i",
                lambda t, x, mu=mu, i=i: jnp.concatenate(
                    [
                        jnp.array([0.0], dtype=jnp.float32),
                        jnp.zeros((D,), dtype=jnp.float32).at[i].set(
                            safe_exp(mu * time_basis(t))
                        ),
                    ]
                ),
            )

        # e^{2mu t/T}(∂t + mu x·∇x)
        add_basis(
            f"exp_projective__mu_{mu:+.2f}__e_2mut_dt_plus_mu_x_dot_dx",
            "Gaeta-style projective linear-drift generator",
            lambda t, x, mu=mu: jnp.concatenate(
                [
                    jnp.array(
                        [safe_exp(2.0 * mu * time_basis(t))],
                        dtype=jnp.float32,
                    ),
                    mu * safe_exp(2.0 * mu * time_basis(t)) * x,
                ]
            ),
        )


# ============================================================
# Group F: low-rank rotations and shears
# ============================================================

LOW_RANK_DIRECTIONS = make_low_rank_directions(CONFIG["BASIS_LOW_RANK_K"])
K_LOW = len(LOW_RANK_DIRECTIONS)

print(f"Using {K_LOW} fixed low-rank basis directions.")


# ----------------------------
# Low-rank rotations:
#   (v_b·x)v_a - (v_a·x)v_b
# ----------------------------
if CONFIG.get("BASIS_ADD_LOW_RANK_ROTATIONS", False):
    for a in range(K_LOW):
        for b in range(a + 1, K_LOW):
            va = LOW_RANK_DIRECTIONS[a]
            vb = LOW_RANK_DIRECTIONS[b]

            add_basis(
                f"lowrank_rotation__v{a+1}_v{b+1}",
                "Low-rank antisymmetric rotation generator",
                lambda t, x, va=va, vb=vb: jnp.concatenate(
                    [
                        jnp.array([0.0], dtype=jnp.float32),
                        jnp.dot(vb, x) * va - jnp.dot(va, x) * vb,
                    ]
                ),
            )


# ----------------------------
# Low-rank adjacent shears:
#   (v_b·x)v_a
#
# We use only adjacent directions and both orientations:
#   v_{a+1} -> v_a and v_a -> v_{a+1}
#
# This adds 2(K-1) shears instead of all K(K-1).
# ----------------------------
if CONFIG.get("BASIS_ADD_LOW_RANK_SHEARS", False):
    for a in range(K_LOW - 1):
        va = LOW_RANK_DIRECTIONS[a]
        vb = LOW_RANK_DIRECTIONS[a + 1]

        add_basis(
            f"lowrank_shear__v{a+2}_to_v{a+1}",
            "Low-rank adjacent shear generator",
            lambda t, x, va=va, vb=vb: jnp.concatenate(
                [
                    jnp.array([0.0], dtype=jnp.float32),
                    jnp.dot(vb, x) * va,
                ]
            ),
        )

        add_basis(
            f"lowrank_shear__v{a+1}_to_v{a+2}",
            "Low-rank adjacent shear generator",
            lambda t, x, va=va, vb=vb: jnp.concatenate(
                [
                    jnp.array([0.0], dtype=jnp.float32),
                    jnp.dot(va, x) * vb,
                ]
            ),
        )


# ============================================================
# Finalize basis library
# ============================================================

BASIS_NAMES = [b["name"] for b in BASIS_LIBRARY]
BASIS_SOURCES = [b["source"] for b in BASIS_LIBRARY]
BASIS_FNS = tuple(b["fn"] for b in BASIS_LIBRARY)
N_BASIS = len(BASIS_FNS)

print(f"\nNumber of 50D basis functions: {N_BASIS}")
print(f"BASIS_MODE: {CONFIG['BASIS_MODE']}")

n_minimal = 3 + D + D + 1
n_exp = 0
if CONFIG.get("BASIS_ADD_EXPONENTIAL", False):
    n_exp = len([mu for mu in CONFIG["BASIS_LINEAR_MU_LIST"] if abs(mu) > 1e-12]) * (D + 1)

n_rot = 0
if CONFIG.get("BASIS_ADD_LOW_RANK_ROTATIONS", False):
    n_rot = K_LOW * (K_LOW - 1) // 2

n_shear = 0
if CONFIG.get("BASIS_ADD_LOW_RANK_SHEARS", False):
    n_shear = 2 * (K_LOW - 1)

print("\nBasis count breakdown:")
print(f"  minimal time/translation/scaling/global : {n_minimal}")
print(f"  exponential Gaeta terms                 : {n_exp}")
print(f"  low-rank rotations                      : {n_rot}")
print(f"  low-rank shears                         : {n_shear}")
print(f"  total                                   : {N_BASIS}")

for i, (nm, src) in enumerate(zip(BASIS_NAMES, BASIS_SOURCES)):
    print(f"{i:03d} | {nm} | {src}")


def eval_basis_one(t, x):
    """
    Returns basis values at one point.

    Output shape:
        (R, 51)
    """
    return jnp.stack([fn(t, x) for fn in BASIS_FNS], axis=0)


def eval_generators_point(W, z):
    """
    Evaluate learned generators at one point.

    W shape:
        (m, R)

    z shape:
        (51,) = [t, x1, ..., x50]

    Returns:
        X shape (m, 51), with X[a] = [tau_a, xi_a1, ..., xi_a50].
    """
    t = z[0]
    x = z[1:]
    B = eval_basis_one(t, x)
    return W @ B


def eval_generators_batch(W, tx):
    """
    tx shape:
        (S, 51)

    Returns:
        X shape (S, m, 51)
    """
    return jax.vmap(lambda z: eval_generators_point(W, z))(tx)


def _x_box_to_arrays(x_box):
    """
    Allows x_box to be either:
        (low_scalar, high_scalar)
    or:
        (low_vector_shape_D, high_vector_shape_D)
    """
    low, high = x_box

    low = jnp.asarray(low, dtype=jnp.float32)
    high = jnp.asarray(high, dtype=jnp.float32)

    if low.ndim == 0:
        low = jnp.full((D,), low, dtype=jnp.float32)

    if high.ndim == 0:
        high = jnp.full((D,), high, dtype=jnp.float32)

    if low.shape != (D,) or high.shape != (D,):
        raise ValueError(
            f"x_box low/high must be scalar or shape ({D},). "
            f"Got {low.shape}, {high.shape}."
        )

    return low, high


def sample_tx_box(key, n_points, t_box, x_box):
    """
    Sample collocation points in:
        [t_min,t_max] x x_box^50

    Output:
        tx shape (n_points, 51)
    """
    key_t, key_x = jax.random.split(key)

    t = jax.random.uniform(
        key_t,
        (n_points, 1),
        minval=float(t_box[0]),
        maxval=float(t_box[1]),
    )

    x_low, x_high = _x_box_to_arrays(x_box)

    u = jax.random.uniform(key_x, (n_points, D))
    x = x_low[None, :] + u * (x_high[None, :] - x_low[None, :])

    return jnp.concatenate([t, x], axis=1)

def sample_tx_data_jitter(key, dataset, n_points, jitter_std=0.05):
    """
    Sample symmetry collocation points near the observed trajectory manifold.
    """
    key_idx, key_noise = jax.random.split(key)

    t0, x0, _, _ = dataset
    n = t0.shape[0]

    idx = jax.random.randint(key_idx, (n_points,), 0, n)

    t = t0[idx]
    x = x0[idx]

    noise = jitter_std * jax.random.normal(key_noise, x.shape)
    x_jitter = x + noise

    return jnp.concatenate([t[:, None], x_jitter], axis=1)


key, subkey = jax.random.split(key)
W = CONFIG["SYM_INIT_SCALE"] * jax.random.normal(
    subkey,
    (CONFIG["N_GENERATORS"], N_BASIS),
)

print("\nInitial W shape:", W.shape)

## Cell 5 — 50D determining loss, global Lie closure, symmetry discovery

In [ ]:
# ============================================================
# Cell 5 — 50D determining loss, global Lie closure, symmetry discovery
# ============================================================

def det_residuals_point(sde_params, W, z):
    """
    50D projectable Ito SDE determining equations.

    Coordinates:
        z = (t, x1, ..., x50)

    Generator:
        X_a = tau_a ∂t + xi_a^i ∂_{x_i}

    Neural SDE:
        dX^i = f^i(t,X) dt + sigma_i(t,X) dW^i

    Diagonal diffusion:
        Sigma^{i alpha} = delta_{i alpha} sigma_i

    Important:
        For the minimal 104-basis, xi is at most linear in x.
        Therefore xi_xx = 0 and the Ito second-derivative term
        in the drift determining equation vanishes.
    """
    f = f_vec_single(sde_params, z)            # (D,)
    sigma = sigma_vec_single(sde_params, z)    # (D,)

    J_f = jax.jacfwd(lambda zz: f_vec_single(sde_params, zz))(z)          # (D, Q)
    J_sigma = jax.jacfwd(lambda zz: sigma_vec_single(sde_params, zz))(z)  # (D, Q)

    f_t = J_f[:, 0]          # (D,)
    f_x = J_f[:, 1:]         # (D, D)

    sigma_t = J_sigma[:, 0]  # (D,)
    sigma_x = J_sigma[:, 1:] # (D, D)

    gen_fun = lambda zz: eval_generators_point(W, zz)

    X = gen_fun(z)                         # (m, Q)
    J = jax.jacfwd(gen_fun)(z)             # (m, Q, Q)

    tau = X[:, 0]                          # (m,)
    xi = X[:, 1:]                          # (m, D)

    tau_t = J[:, 0, 0]                     # (m,)
    tau_x = J[:, 0, 1:]                    # (m, D)

    xi_t = J[:, 1:, 0]                     # (m, D)
    xi_x = J[:, 1:, 1:]                    # (m, D, D)

    adv_xi = jnp.einsum("j,aij->ai", f, xi_x)
    xi_dot_grad_f = jnp.einsum("aj,ij->ai", xi, f_x)
    time_f_tau = tau[:, None] * f_t[None, :] + tau_t[:, None] * f[None, :]

    # Since xi_xx = 0 for the minimal linear basis:
    ito_xi = 0.0

    R_f = (
        xi_t
        + adv_xi
        - xi_dot_grad_f
        - time_f_tau
        + ito_xi
    )  # (m, D)

    # Diffusion residual for diagonal Sigma.
    eye = jnp.eye(D, dtype=jnp.float32)

    first_term = jnp.einsum("p,aip->aip", sigma, xi_x)

    diagonal_correction = (
        jnp.einsum("aj,ij->ai", xi, sigma_x)
        + tau[:, None] * sigma_t[None, :]
        + 0.5 * tau_t[:, None] * sigma[None, :]
    )  # (m, D)

    R_sigma = first_term - diagonal_correction[:, :, None] * eye[None, :, :]

    return R_f, R_sigma, tau_x


def determining_loss(sde_params, W, tx):
    R_f, R_sigma, tau_x = jax.vmap(
        lambda z: det_residuals_point(sde_params, W, z)
    )(tx)

    det = jnp.mean(R_f**2) + jnp.mean(R_sigma**2)
    projectable = jnp.mean(tau_x**2)
    return det, projectable


def generator_jacobians_batch(W, tx):
    """
    Returns:
        X shape (S, m, Q)
        J shape (S, m, Q, Q)
    """
    def one(z):
        gen_fun = lambda zz: eval_generators_point(W, zz)
        X = gen_fun(z)
        J = jax.jacfwd(gen_fun)(z)
        return X, J

    return jax.vmap(one)(tx)


def lie_closure_loss_global(W, tx):
    """
    Global-batch Lie closure in coordinates (t,x1,x2,x3).

        [X_i, X_j] ≈ sum_k C_ij^k X_k

    Structure constants are global because we solve a single least-squares
    problem over the whole collocation batch.
    """
    m = W.shape[0]
    if m < 2:
        return jnp.array(0.0, dtype=jnp.float32)

    X, J = generator_jacobians_batch(W, tx)
    S = tx.shape[0]

    idx_i, idx_j = jnp.triu_indices(m, k=1)
    P = idx_i.shape[0]

    Xi = X[:, idx_i, :]       # (S, P, Q)
    Xj = X[:, idx_j, :]       # (S, P, Q)
    Ji = J[:, idx_i, :, :]    # (S, P, Q, Q)
    Jj = J[:, idx_j, :, :]    # (S, P, Q, Q)

    brackets = (
        jnp.einsum("spc,spqc->spq", Xi, Jj)
        - jnp.einsum("spc,spqc->spq", Xj, Ji)
    )  # (S, P, Q)

    V = jnp.transpose(X, (1, 0, 2)).reshape(m, -1).T      # (Q*S, m)
    B = jnp.transpose(brackets, (1, 0, 2)).reshape(P, -1).T  # (Q*S, P)

    G = V.T @ V
    reg = (CONFIG["CLOSURE_RCOND"] ** 2) * jnp.eye(m, dtype=G.dtype)
    rhs = V.T @ B

    C = jnp.linalg.solve(G + reg, rhs)
    B_proj = V @ C
    residual = B - B_proj

    return jnp.mean(residual**2)


def gram_independence_loss(W, tx):
    """
    Function-space Gram orthonormality:
        G_ab = E[tau_a tau_b + xi_a · xi_b]
    """
    X = eval_generators_batch(W, tx)  # (S, m, Q)
    S = X.shape[0]
    m = X.shape[1]

    V = jnp.transpose(X, (1, 0, 2)).reshape(m, -1).T  # (Q*S, m)
    G = (V.T @ V) / (Q * S)

    I = jnp.eye(m, dtype=G.dtype)
    loss_ind = jnp.mean((G - I) ** 2)

    diag = jnp.diag(G)
    norm_hinge = jnp.mean(
        jnp.maximum(0.0, CONFIG["MIN_GENERATOR_NORM"] - diag) ** 2
    )

    return loss_ind, norm_hinge, G


def sym_discovery_objective(W, sde_params, tx):
    det, proj = determining_loss(sde_params, W, tx)
    close = lie_closure_loss_global(W, tx)
    ind, norm_hinge, _ = gram_independence_loss(W, tx)

    total = (
        CONFIG["LAMBDA_DET"] * det
        + CONFIG["LAMBDA_CLOSE"] * close
        + CONFIG["LAMBDA_IND"] * ind
        + CONFIG["LAMBDA_PROJECTABLE"] * proj
        + CONFIG["LAMBDA_NORM_HINGE"] * norm_hinge
    )

    aux = {
        "total": total,
        "det": det,
        "close": close,
        "ind": ind,
        "projectable": proj,
        "norm_hinge": norm_hinge,
    }
    return total, aux


sym_value_and_grad = jax.value_and_grad(
    lambda W, sde_params, tx: sym_discovery_objective(W, sde_params, tx)[0]
)


@jax.jit
def sym_train_step(W, opt_state, sde_params, tx):
    loss, grads = sym_value_and_grad(W, sde_params, tx)
    W, opt_state = adam_update(
        W,
        grads,
        opt_state,
        CONFIG["SYM_DISCOVERY_LR"],
    )
    _, aux = sym_discovery_objective(W, sde_params, tx)
    return W, opt_state, aux


# ----------------------------
# Train symmetry basis weights W
# ----------------------------
W_opt_state = adam_init(W)
sym_history = []

t_start = time.time()

for epoch in range(1, CONFIG["SYM_DISCOVERY_EPOCHS"] + 1):
    key, subkey = jax.random.split(key)
    tx_disc = sample_tx_data_jitter(
        subkey,
        train_data,
        CONFIG["SYM_DISCOVERY_POINTS"],
        jitter_std=0.05,
    )

    W, W_opt_state, aux = sym_train_step(W, W_opt_state, sde_params, tx_disc)

    if epoch % CONFIG["PRINT_EVERY"] == 0 or epoch == 1:
        aux_np = {k: float(v) for k, v in aux.items()}
        sym_history.append(aux_np)
        print(
            f"[SymDiscovery] epoch {epoch:5d} | "
            f"total {aux_np['total']: .4e} | "
            f"det {aux_np['det']: .4e} | "
            f"close {aux_np['close']: .4e} | "
            f"ind {aux_np['ind']: .4e} | "
            f"proj {aux_np['projectable']: .4e}"
        )

print(f"Symmetry discovery time: {time.time() - t_start:.2f} sec")

W_np = np.array(W)
print("\nLearned W shape:", W_np.shape)

for a in range(CONFIG["N_GENERATORS"]):
    print(f"\nGenerator X_{a+1}: top basis coefficients")
    order = np.argsort(-np.abs(W_np[a]))
    for idx in order[:min(10, N_BASIS)]:
        print(f"  {idx:02d} | coeff {W_np[a, idx]: .5f} | {BASIS_NAMES[idx]}")

# Evaluate generator Gram matrix on near-trajectory points,
# consistent with how W was discovered.
key, subkey = jax.random.split(key)
tx_eval_sym = sample_tx_data_jitter(
    subkey,
    train_data,
    256,
    jitter_std=0.05,
)

_, _, G_eval = gram_independence_loss(W, tx_eval_sym)
print("\nGenerator Gram matrix on near-trajectory jittered eval points:")
print(np.array(G_eval))

## Cell 6 — Stage F: symmetry-regularized 50D surrogate refinement

In [ ]:
# ============================================================
# Cell 6 — Stage F: symmetry-regularized 50D surrogate refinement
# ============================================================

def stage_f_objective(sde_params, W_fixed, batch, tx_extra):
    """
    Stage F objective for 50D SDE.

    Symmetry loss is evaluated on:
        1. a small subset of trajectory batch points
        2. extra collocation points sampled from the 51D tx box
    """
    data_loss = data_nll_loss(sde_params, batch)

    t_batch, x_batch, _, _ = batch

    n_data_sym = min(
        CONFIG["STAGE_F_N_DATA_SYM_POINTS"],
        t_batch.shape[0],
    )

    tx_data = jnp.concatenate(
        [t_batch[:n_data_sym, None], x_batch[:n_data_sym]],
        axis=1,
    )

    tx_sym = jnp.concatenate([tx_data, tx_extra], axis=0)

    det, proj = determining_loss(sde_params, W_fixed, tx_sym)

    total = (
        data_loss
        + CONFIG["STAGE_F_LAMBDA_SYM_DET"] * det
        + CONFIG["STAGE_F_LAMBDA_PROJECTABLE"] * proj
    )

    aux = {
        "total": total,
        "data": data_loss,
        "sym_det": det,
        "projectable": proj,
    }
    return total, aux


stage_f_value_and_grad = jax.value_and_grad(
    lambda params, W_fixed, batch, tx_extra: stage_f_objective(
        params,
        W_fixed,
        batch,
        tx_extra,
    )[0]
)


@jax.jit
def stage_f_train_step(sde_params, opt_state, W_fixed, batch, tx_extra):
    loss, grads = stage_f_value_and_grad(sde_params, W_fixed, batch, tx_extra)
    sde_params, opt_state = adam_update(
        sde_params,
        grads,
        opt_state,
        CONFIG["STAGE_F_LR"],
    )
    _, aux = stage_f_objective(sde_params, W_fixed, batch, tx_extra)
    return sde_params, opt_state, aux


# Save baseline params for comparison
sde_params_baseline = sde_params

# Start Stage F from baseline params
sde_params_refined = sde_params
stage_f_opt_state = adam_init(sde_params_refined)

stage_f_history = []
t_start = time.time()

for epoch in range(1, CONFIG["STAGE_F_EPOCHS"] + 1):
    key, key_batch, key_extra = jax.random.split(key, 3)

    batch = sample_batch(
        key_batch,
        train_data,
        CONFIG["STAGE_F_BATCH_SIZE"],
    )

    tx_extra = sample_tx_data_jitter(
        key_extra,
        train_data,
        CONFIG["STAGE_F_N_EXTRA_SYM_POINTS"],
        jitter_std=0.05,
    )

    sde_params_refined, stage_f_opt_state, aux = stage_f_train_step(
        sde_params_refined,
        stage_f_opt_state,
        W,
        batch,
        tx_extra,
    )

    if epoch % CONFIG["PRINT_EVERY"] == 0 or epoch == 1:
        aux_np = {k: float(v) for k, v in aux.items()}
        stage_f_history.append(aux_np)

        train_nll = float(data_nll_loss(sde_params_refined, train_data))
        test_nll = float(data_nll_loss(sde_params_refined, test_data))

        print(
            f"[Stage F] epoch {epoch:5d} | "
            f"total {aux_np['total']: .5f} | "
            f"data {aux_np['data']: .5f} | "
            f"sym_det {aux_np['sym_det']: .4e} | "
            f"train nll {train_nll: .5f} | "
            f"test nll {test_nll: .5f}"
        )

print(f"Stage F training time: {time.time() - t_start:.2f} sec")

baseline_train_nll = float(data_nll_loss(sde_params_baseline, train_data))
baseline_test_nll = float(data_nll_loss(sde_params_baseline, test_data))
refined_train_nll = float(data_nll_loss(sde_params_refined, train_data))
refined_test_nll = float(data_nll_loss(sde_params_refined, test_data))

print("\n================ Final NLL Comparison ================")
print(f"Baseline train NLL: {baseline_train_nll: .5f}")
print(f"Baseline test  NLL: {baseline_test_nll: .5f}")
print(f"Refined  train NLL: {refined_train_nll: .5f}")
print(f"Refined  test  NLL: {refined_test_nll: .5f}")

# ============================================================
# Final symmetry residual comparison on multiple point distributions
# ============================================================

def residual_pair(tx):
    det_b, proj_b = determining_loss(sde_params_baseline, W, tx)
    det_r, proj_r = determining_loss(sde_params_refined, W, tx)
    return float(det_b), float(det_r), float(proj_b), float(proj_r)


key, key_box, key_train, key_jitter = jax.random.split(key, 4)

# 1. Uniform box points: mostly off-manifold in 50D.
tx_sym_box = sample_tx_box(
    key_box,
    256,
    CONFIG["STAGE_F_EXTRA_T_BOX"],
    CONFIG["STAGE_F_EXTRA_X_BOX"],
)

# 2. Actual training trajectory states.
batch_eval = sample_batch(key_train, train_data, 256)
t_eval, x_eval, _, _ = batch_eval
tx_sym_train = jnp.concatenate([t_eval[:, None], x_eval], axis=1)

# 3. Near-trajectory jittered points: same distribution as Stage F extra points.
tx_sym_jitter = sample_tx_data_jitter(
    key_jitter,
    train_data,
    256,
    jitter_std=0.05,
)

box_det_b, box_det_r, box_proj_b, box_proj_r = residual_pair(tx_sym_box)
train_det_b, train_det_r, train_proj_b, train_proj_r = residual_pair(tx_sym_train)
jit_det_b, jit_det_r, jit_proj_b, jit_proj_r = residual_pair(tx_sym_jitter)

print("\n============ Symmetry Residual Comparison ============")
print("Lower is better. Positive reduction means refined improved.")

print("\n[Uniform 50D box points: off-manifold diagnostic]")
print(f"Baseline det residual: {box_det_b: .4e}")
print(f"Refined  det residual: {box_det_r: .4e}")
print(f"Det residual reduction: {100.0 * (box_det_b - box_det_r) / (abs(box_det_b) + 1e-12): .2f}%")
print(f"Baseline projectability residual: {box_proj_b: .4e}")
print(f"Refined  projectability residual: {box_proj_r: .4e}")

print("\n[Actual train trajectory points: on-manifold diagnostic]")
print(f"Baseline det residual: {train_det_b: .4e}")
print(f"Refined  det residual: {train_det_r: .4e}")
print(f"Det residual reduction: {100.0 * (train_det_b - train_det_r) / (abs(train_det_b) + 1e-12): .2f}%")
print(f"Baseline projectability residual: {train_proj_b: .4e}")
print(f"Refined  projectability residual: {train_proj_r: .4e}")

print("\n[Near-trajectory jittered points: Stage-F-matched diagnostic]")
print(f"Baseline det residual: {jit_det_b: .4e}")
print(f"Refined  det residual: {jit_det_r: .4e}")
print(f"Det residual reduction: {100.0 * (jit_det_b - jit_det_r) / (abs(jit_det_b) + 1e-12): .2f}%")
print(f"Baseline projectability residual: {jit_proj_b: .4e}")
print(f"Refined  projectability residual: {jit_proj_r: .4e}")

## Cell 7 — 50D diagnostics, numerical comparisons, and plots

In [ ]:
# ============================================================
# Cell 7 — 50D diagnostics, numerical comparisons, and plots
# ============================================================

print("\n" + "=" * 72)
print("DIAGNOSTICS FOR ACTIVE 50D SDE")
print("=" * 72)
print(f"TRUE_SDE_NAME: {CONFIG['TRUE_SDE_NAME']}")
print(f"Description  : {ACTIVE_SDE['description']}")
print(f"STATE_DIM    : {D}")
print(f"T_FINAL      : {CONFIG['T_FINAL']}")
print(f"X_BOX        : {CONFIG['STAGE_F_EXTRA_X_BOX']}")
print("=" * 72)

def batched_surrogate_values(params, t_values, x_values):
    """
    Evaluate f_theta and sigma_theta on paired arrays.

    t_values shape:
        (S,)

    x_values shape:
        (S, 3)

    Returns:
        f shape (S, 3)
        sigma shape (S, 3)
    """
    f, s = jax.vmap(lambda tt, xx: surrogate_f_sigma(params, tt, xx))(t_values, x_values)
    return f, s


def safe_percent_change(new, old, eps=1e-12):
    return 100.0 * (new - old) / (abs(old) + eps)


def safe_percent_reduction(new, old, eps=1e-12):
    return 100.0 * (old - new) / (abs(old) + eps)


def rmse(a, b):
    return jnp.sqrt(jnp.mean((a - b) ** 2))


def mae(a, b):
    return jnp.mean(jnp.abs(a - b))


def mse(a, b):
    return jnp.mean((a - b) ** 2)


def relative_l2_error(pred, truth, eps=1e-12):
    return jnp.sqrt(jnp.sum((pred - truth) ** 2) / (jnp.sum(truth ** 2) + eps))


def one_step_prediction_diagnostics(params, dataset):
    """
    3D one-step Euler-Maruyama increment diagnostics.
    """
    t0, x0, dt, dx = dataset
    f, s = jax.vmap(lambda tt, xx: surrogate_f_sigma(params, tt, xx))(t0, x0)

    mean_pred = f * dt[:, None]
    var_pred = s**2 * dt[:, None] + 1e-8

    inc_rmse = rmse(mean_pred, dx)
    inc_mae = mae(mean_pred, dx)

    standardized_sq_residual = jnp.mean((dx - mean_pred) ** 2 / var_pred)
    avg_pred_std = jnp.mean(jnp.sqrt(var_pred))
    avg_abs_dx = jnp.mean(jnp.abs(dx))

    return {
        "increment_rmse": float(inc_rmse),
        "increment_mae": float(inc_mae),
        "standardized_sq_residual_mean": float(standardized_sq_residual),
        "avg_pred_step_std": float(avg_pred_std),
        "avg_abs_dx": float(avg_abs_dx),
    }


def coefficient_metrics_on_random_box(params, n_points, t_box, x_box):
    key_local = jax.random.PRNGKey(12345)
    tx = sample_tx_box(key_local, n_points, t_box, x_box)

    t = tx[:, 0]
    x = tx[:, 1:]

    f_true = jax.vmap(lambda tt, xx: true_drift(tt, xx))(t, x)
    s_true = jax.vmap(lambda tt, xx: true_sigma(tt, xx))(t, x)

    f_pred, s_pred = batched_surrogate_values(params, t, x)

    return {
        "drift_mse": float(mse(f_pred, f_true)),
        "drift_rmse": float(rmse(f_pred, f_true)),
        "drift_mae": float(mae(f_pred, f_true)),
        "drift_rel_l2": float(relative_l2_error(f_pred, f_true)),
        "sigma_mse": float(mse(s_pred, s_true)),
        "sigma_rmse": float(rmse(s_pred, s_true)),
        "sigma_mae": float(mae(s_pred, s_true)),
        "sigma_rel_l2": float(relative_l2_error(s_pred, s_true)),
    }


# ============================================================
# Numerical diagnostics
# ============================================================

base_coeff = coefficient_metrics_on_random_box(
    sde_params_baseline,
    CONFIG["N_EVAL_POINTS"],
    CONFIG["STAGE_F_EXTRA_T_BOX"],
    CONFIG["STAGE_F_EXTRA_X_BOX"],
)

ref_coeff = coefficient_metrics_on_random_box(
    sde_params_refined,
    CONFIG["N_EVAL_POINTS"],
    CONFIG["STAGE_F_EXTRA_T_BOX"],
    CONFIG["STAGE_F_EXTRA_X_BOX"],
)

baseline_train_nll = float(data_nll_loss(sde_params_baseline, train_data))
baseline_test_nll = float(data_nll_loss(sde_params_baseline, test_data))

refined_train_nll = float(data_nll_loss(sde_params_refined, train_data))
refined_test_nll = float(data_nll_loss(sde_params_refined, test_data))

# ============================================================
# Symmetry residual diagnostics on different point distributions
# ============================================================

def residual_pair(tx):
    det_b, proj_b = determining_loss(sde_params_baseline, W, tx)
    det_r, proj_r = determining_loss(sde_params_refined, W, tx)
    return float(det_b), float(det_r), float(proj_b), float(proj_r)


key, key_box, key_train, key_jitter = jax.random.split(key, 4)

# 1. Uniform box points: useful global/off-manifold diagnostic.
tx_sym_box = sample_tx_box(
    key_box,
    256,
    CONFIG["STAGE_F_EXTRA_T_BOX"],
    CONFIG["STAGE_F_EXTRA_X_BOX"],
)

# 2. Actual train trajectory points.
batch_eval = sample_batch(key_train, train_data, 256)
t_eval, x_eval, _, _ = batch_eval
tx_sym_train = jnp.concatenate([t_eval[:, None], x_eval], axis=1)

# 3. Near-trajectory jittered points: matches Stage F extra-point distribution.
tx_sym_jitter = sample_tx_data_jitter(
    key_jitter,
    train_data,
    256,
    jitter_std=0.05,
)

box_det_base, box_det_ref, box_proj_base, box_proj_ref = residual_pair(tx_sym_box)
train_det_base, train_det_ref, train_proj_base, train_proj_ref = residual_pair(tx_sym_train)
jit_det_base, jit_det_ref, jit_proj_base, jit_proj_ref = residual_pair(tx_sym_jitter)

# Primary symmetry numbers used later in compact reports:
# use the Stage-F-matched near-trajectory jittered diagnostic.
det_base = jit_det_base
det_ref = jit_det_ref
proj_base = jit_proj_base
proj_ref = jit_proj_ref

train_step_base = one_step_prediction_diagnostics(sde_params_baseline, train_data)
train_step_ref = one_step_prediction_diagnostics(sde_params_refined, train_data)

test_step_base = one_step_prediction_diagnostics(sde_params_baseline, test_data)
test_step_ref = one_step_prediction_diagnostics(sde_params_refined, test_data)


# ============================================================
# Print numerical diagnostics
# ============================================================

print("\n" + "=" * 72)
print("50D NUMERICAL COMPARISON: BASELINE vs GAETA-LIE REFINED")
print("=" * 72)

print("\n[1] Negative log-likelihood")
print(f"Train NLL baseline : {baseline_train_nll: .6f}")
print(f"Train NLL refined  : {refined_train_nll: .6f}")
print(f"Train NLL change   : {refined_train_nll - baseline_train_nll: .6f}")
print(f"Train NLL % change : {safe_percent_change(refined_train_nll, baseline_train_nll): .2f}%")

print(f"\nTest NLL baseline  : {baseline_test_nll: .6f}")
print(f"Test NLL refined   : {refined_test_nll: .6f}")
print(f"Test NLL change    : {refined_test_nll - baseline_test_nll: .6f}")
print(f"Test NLL % change  : {safe_percent_change(refined_test_nll, baseline_test_nll): .2f}%")


print("\n[2] Drift reconstruction over random 50D tx-box")
print(f"Drift MSE baseline        : {base_coeff['drift_mse']: .6e}")
print(f"Drift MSE refined         : {ref_coeff['drift_mse']: .6e}")
print(f"Drift MSE reduction       : {safe_percent_reduction(ref_coeff['drift_mse'], base_coeff['drift_mse']): .2f}%")

print(f"\nDrift RMSE baseline       : {base_coeff['drift_rmse']: .6e}")
print(f"Drift RMSE refined        : {ref_coeff['drift_rmse']: .6e}")
print(f"Drift RMSE reduction      : {safe_percent_reduction(ref_coeff['drift_rmse'], base_coeff['drift_rmse']): .2f}%")

print(f"\nDrift relative L2 baseline: {base_coeff['drift_rel_l2']: .6e}")
print(f"Drift relative L2 refined : {ref_coeff['drift_rel_l2']: .6e}")
print(f"Drift relative L2 reduction: {safe_percent_reduction(ref_coeff['drift_rel_l2'], base_coeff['drift_rel_l2']): .2f}%")


print("\n[3] Diffusion reconstruction over random 4D tx-box")
print(f"Sigma MSE baseline        : {base_coeff['sigma_mse']: .6e}")
print(f"Sigma MSE refined         : {ref_coeff['sigma_mse']: .6e}")
print(f"Sigma MSE reduction       : {safe_percent_reduction(ref_coeff['sigma_mse'], base_coeff['sigma_mse']): .2f}%")

print(f"\nSigma RMSE baseline       : {base_coeff['sigma_rmse']: .6e}")
print(f"Sigma RMSE refined        : {ref_coeff['sigma_rmse']: .6e}")
print(f"Sigma RMSE reduction      : {safe_percent_reduction(ref_coeff['sigma_rmse'], base_coeff['sigma_rmse']): .2f}%")

print(f"\nSigma relative L2 baseline: {base_coeff['sigma_rel_l2']: .6e}")
print(f"Sigma relative L2 refined : {ref_coeff['sigma_rel_l2']: .6e}")
print(f"Sigma relative L2 reduction: {safe_percent_reduction(ref_coeff['sigma_rel_l2'], base_coeff['sigma_rel_l2']): .2f}%")


print("\n[4] Symmetry determining-equation residual")
print("Lower is better. Positive reduction means refined improved.")

print("\n[4a] Uniform 50D box points: off-manifold/global diagnostic")
print(f"Baseline determining residual : {box_det_base: .6e}")
print(f"Refined determining residual  : {box_det_ref: .6e}")
print(f"Residual reduction            : {safe_percent_reduction(box_det_ref, box_det_base): .2f}%")
print(f"Baseline projectability residual : {box_proj_base: .6e}")
print(f"Refined projectability residual  : {box_proj_ref: .6e}")
print(f"Projectability residual reduction: {safe_percent_reduction(box_proj_ref, box_proj_base): .2f}%")

print("\n[4b] Actual train trajectory points: on-manifold diagnostic")
print(f"Baseline determining residual : {train_det_base: .6e}")
print(f"Refined determining residual  : {train_det_ref: .6e}")
print(f"Residual reduction            : {safe_percent_reduction(train_det_ref, train_det_base): .2f}%")
print(f"Baseline projectability residual : {train_proj_base: .6e}")
print(f"Refined projectability residual  : {train_proj_ref: .6e}")
print(f"Projectability residual reduction: {safe_percent_reduction(train_proj_ref, train_proj_base): .2f}%")

print("\n[4c] Near-trajectory jittered points: Stage-F-matched diagnostic")
print(f"Baseline determining residual : {jit_det_base: .6e}")
print(f"Refined determining residual  : {jit_det_ref: .6e}")
print(f"Residual reduction            : {safe_percent_reduction(jit_det_ref, jit_det_base): .2f}%")
print(f"Baseline projectability residual : {jit_proj_base: .6e}")
print(f"Refined projectability residual  : {jit_proj_ref: .6e}")
print(f"Projectability residual reduction: {safe_percent_reduction(jit_proj_ref, jit_proj_base): .2f}%")


print("\n[5] One-step increment prediction diagnostics: train")
print(f"Increment RMSE baseline : {train_step_base['increment_rmse']: .6e}")
print(f"Increment RMSE refined  : {train_step_ref['increment_rmse']: .6e}")
print(f"RMSE reduction          : {safe_percent_reduction(train_step_ref['increment_rmse'], train_step_base['increment_rmse']): .2f}%")
print(f"Mean standardized squared residual baseline : {train_step_base['standardized_sq_residual_mean']: .6f}")
print(f"Mean standardized squared residual refined  : {train_step_ref['standardized_sq_residual_mean']: .6f}")


print("\n[6] One-step increment prediction diagnostics: test")
print(f"Increment RMSE baseline : {test_step_base['increment_rmse']: .6e}")
print(f"Increment RMSE refined  : {test_step_ref['increment_rmse']: .6e}")
print(f"RMSE reduction          : {safe_percent_reduction(test_step_ref['increment_rmse'], test_step_base['increment_rmse']): .2f}%")
print(f"Mean standardized squared residual baseline : {test_step_base['standardized_sq_residual_mean']: .6f}")
print(f"Mean standardized squared residual refined  : {test_step_ref['standardized_sq_residual_mean']: .6f}")


# ============================================================
# Plot training curves
# ============================================================

plt.figure(figsize=(8, 4))
plt.plot(baseline_losses)
plt.xlabel("baseline epoch")
plt.ylabel("batch NLL")
plt.title("Baseline 3D neural SDE training")
plt.grid(True)
plt.show()


if len(sym_history) > 0:
    plt.figure(figsize=(8, 4))
    plt.plot([h["det"] for h in sym_history], label="determining")
    plt.plot([h["close"] for h in sym_history], label="closure")
    plt.plot([h["ind"] for h in sym_history], label="independence")
    plt.yscale("log")
    plt.xlabel(f"checkpoint every {CONFIG['PRINT_EVERY']} epochs")
    plt.ylabel("loss")
    plt.title("3D symmetry discovery losses")
    plt.legend()
    plt.grid(True)
    plt.show()


if len(stage_f_history) > 0:
    plt.figure(figsize=(8, 4))
    plt.plot([h["data"] for h in stage_f_history], label="data NLL")
    plt.plot([h["sym_det"] for h in stage_f_history], label="sym det")
    plt.yscale("log")
    plt.xlabel(f"checkpoint every {CONFIG['PRINT_EVERY']} epochs")
    plt.ylabel("loss")
    plt.title("3D Stage F losses")
    plt.legend()
    plt.grid(True)
    plt.show()


# ============================================================
# 1D diagnostic slices through 50D state space
# Vary x1 while fixing all other coordinates to 0
# Plot only selected components.
# ============================================================

x1_grid = jnp.linspace(
    CONFIG["STAGE_F_EXTRA_X_BOX"][0],
    CONFIG["STAGE_F_EXTRA_X_BOX"][1],
    CONFIG["N_PLOT_GRID"],
)

x_line = jnp.zeros((CONFIG["N_PLOT_GRID"], D), dtype=jnp.float32)
x_line = x_line.at[:, 0].set(x1_grid)

t_slices = [0.0, 0.5 * CONFIG["T_FINAL"], CONFIG["T_FINAL"]]
plot_components = CONFIG["PLOT_COMPONENTS"]

for t0 in t_slices:
    t_grid = jnp.full((CONFIG["N_PLOT_GRID"],), t0)

    f_true = jax.vmap(lambda tt, xx: true_drift(tt, xx))(t_grid, x_line)
    s_true = jax.vmap(lambda tt, xx: true_sigma(tt, xx))(t_grid, x_line)

    f_base, s_base = batched_surrogate_values(sde_params_baseline, t_grid, x_line)
    f_ref, s_ref = batched_surrogate_values(sde_params_refined, t_grid, x_line)

    for comp in plot_components:
        plt.figure(figsize=(8, 4))
        plt.plot(np.array(x1_grid), np.array(f_true[:, comp]), label=f"true f{comp+1}")
        plt.plot(np.array(x1_grid), np.array(f_base[:, comp]), "--", label=f"baseline f{comp+1}")
        plt.plot(np.array(x1_grid), np.array(f_ref[:, comp]), ":", label=f"refined f{comp+1}")
        plt.xlabel("x1, with all other coordinates fixed at 0")
        plt.ylabel(f"f{comp+1}(t,x)")
        plt.title(f"Drift component {comp+1} at t={t0:.2f}")
        plt.legend()
        plt.grid(True)
        plt.show()

        plt.figure(figsize=(8, 4))
        plt.plot(np.array(x1_grid), np.array(s_true[:, comp]), label=f"true sigma{comp+1}")
        plt.plot(np.array(x1_grid), np.array(s_base[:, comp]), "--", label=f"baseline sigma{comp+1}")
        plt.plot(np.array(x1_grid), np.array(s_ref[:, comp]), ":", label=f"refined sigma{comp+1}")
        plt.xlabel("x1, with all other coordinates fixed at 0")
        plt.ylabel(f"sigma{comp+1}(t,x)")
        plt.title(f"Diffusion component {comp+1} at t={t0:.2f}")
        plt.legend()
        plt.grid(True)
        plt.show()


# ============================================================
# Bar plot
# ============================================================

metric_names = [
    "Test NLL",
    "Drift RMSE",
    "Sigma RMSE",
    "Sym Det",
    "Test Inc RMSE",
]

baseline_metric_values = np.array([
    abs(baseline_test_nll),
    base_coeff["drift_rmse"],
    base_coeff["sigma_rmse"],
    det_base,
    test_step_base["increment_rmse"],
], dtype=float)

refined_metric_values = np.array([
    abs(refined_test_nll),
    ref_coeff["drift_rmse"],
    ref_coeff["sigma_rmse"],
    det_ref,
    test_step_ref["increment_rmse"],
], dtype=float)

x_pos = np.arange(len(metric_names))
width = 0.35

plt.figure(figsize=(10, 4))
plt.bar(x_pos - width / 2, baseline_metric_values, width, label="baseline")
plt.bar(x_pos + width / 2, refined_metric_values, width, label="Gaeta-Lie refined")
plt.xticks(x_pos, metric_names, rotation=25, ha="right")
plt.yscale("log")
plt.ylabel("metric value, log scale")
plt.title("3D baseline vs Gaeta-Lie refined numerical diagnostics")
plt.legend()
plt.grid(True, axis="y")
plt.tight_layout()
plt.show()


# ============================================================
# Generator inspection
# ============================================================

W_np = np.array(W)

print("\n" + "=" * 72)
print(f"50D COMPACT FINAL REPORT — {CONFIG['TRUE_SDE_NAME']}")
print("=" * 72)

print(f"TRUE_SDE_NAME: {CONFIG['TRUE_SDE_NAME']}")
print(f"Description: {ACTIVE_SDE['description']}")
print(f"N_GENERATORS used in Stage F: {CONFIG['N_GENERATORS']}")
print("For 50D full-rank diffusion, theoretical bound is n+2=52.")
print("In practice, start with N_GENERATORS = 1, then try 2, 3, 5.")
print(f"N_BASIS in editable 3D basis library: {N_BASIS}")
print(f"Stage F extra symmetry points per step: {CONFIG['STAGE_F_N_EXTRA_SYM_POINTS']}")
print(f"Stage F extra t-box: {CONFIG['STAGE_F_EXTRA_T_BOX']}")
print(f"Stage F extra x-box: {CONFIG['STAGE_F_EXTRA_X_BOX']}")

print("\nNLL:")
print(f"  baseline train: {baseline_train_nll: .6f}")
print(f"  baseline test : {baseline_test_nll: .6f}")
print(f"  refined  train: {refined_train_nll: .6f}")
print(f"  refined  test : {refined_test_nll: .6f}")

print("\nDrift/Diffusion reconstruction over random tx-box:")
print(f"  drift RMSE baseline : {base_coeff['drift_rmse']: .6e}")
print(f"  drift RMSE refined  : {ref_coeff['drift_rmse']: .6e}")
print(f"  sigma RMSE baseline : {base_coeff['sigma_rmse']: .6e}")
print(f"  sigma RMSE refined  : {ref_coeff['sigma_rmse']: .6e}")

print("\nSymmetry determining residual:")
print(f"  baseline: {det_base: .6e}")
print(f"  refined : {det_ref: .6e}")

print("\nTop coefficients per learned generator:")
for a in range(CONFIG["N_GENERATORS"]):
    order = np.argsort(-np.abs(W_np[a]))
    print(f"\nX_{a+1}:")
    for idx in order[:10]:
        print(f"  coeff {W_np[a, idx]: .5f} | {BASIS_NAMES[idx]}")


# Evaluate average generator components on a random cloud.
key, subkey = jax.random.split(key)
tx_vis = sample_tx_box(
    subkey,
    2000,
    CONFIG["SYM_DISCOVERY_T_BOX"],
    CONFIG["SYM_DISCOVERY_X_BOX"],
)

X_vis = eval_generators_batch(W, tx_vis)  # (S, m, 4)
X_vis_np = np.array(X_vis)

print("\nAverage absolute generator components over random tx cloud:")
print("showing |tau| and first 10 |xi_i| components")
for a in range(CONFIG["N_GENERATORS"]):
    avg_abs = np.mean(np.abs(X_vis_np[:, a, :]), axis=0)
    print(f"X_{a+1}: |tau|={avg_abs[0]:.4e}, first 10 |xi|={avg_abs[1:11]}")